# De Novo Protein Binder Design with NVIDIA Proteina-Complexa + Boltz-2

Copyright (c) 2026, NVIDIA CORPORATION. Licensed under the Apache License, Version 2.0 (the "License") you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.

---

**Brev Launchable · optional stream.** This notebook runs an end-to-end de novo **protein binder
design** campaign with **NVIDIA Proteina-Complexa** — a generative model that **co-designs the
binder sequence and full-atom structure together** with reward-guided search — and then
**independently validates** each binder by refolding the complex with **Boltz-2** and ranking on
interface confidence. It is tuned to finish in **well under an hour** on a single **A100 / H100 / L40S** GPU.

> Sibling workflow: `protein-binder-design/` uses **RFdiffusion + ProteinMPNN**. This one uses
> **Proteina-Complexa** (no separate inverse-folding step).

## 1. Why de novo binder design — and why two models

A **binder** is a protein engineered to stick to a target at a chosen site (**hotspots/epitope**).
De novo design invents a brand-new binder (therapeutics, diagnostics, research tools) instead of
screening natural libraries.

This workflow deliberately uses **two independent model families**:

- **Generation — Proteina-Complexa:** co-designs binder **sequence + 3D structure** in one shot. We
  use it to produce a **broad pool** of candidates quickly (no AF2), and it can *optionally* do
  reward-guided **test-time search** (best-of-n / beam / FK-steering / MCTS with an AF2 reward).
- **Assessment — Boltz-2:** a *different* model **co-folds** each binder–target complex (**holo**),
  giving an **independent** interface confidence used to **rank** the pool (optionally also refolding
  the binder alone (**apo**) for a stability check). The validator is not the generator grading its
  own homework.

## 2. Target: SARS-CoV-2 spike RBD (therapeutic relevance)

We design binders against the **Receptor-Binding Domain (RBD)** of the SARS-CoV-2 spike (registered
Complexa target **`30_SC2RBD`**, PDB `6M0J`, target span `E333-526`, conditioned hotspots
`E485, E489, E494, E500, E505`, binder length 80–120 aa).

The RBD grabs the human receptor **ACE2** to enter cells, so a binder that covers this interface
**blocks viral entry** (neutralizing) — the basis of antibody and de novo-minibinder therapeutics
and diagnostics. Swap `TARGET` below for any of the 44 registered targets (`complexa target list`)
— e.g. PD-L1, IL-17A, TNFα, VEGFA.

> Educational example; verify epitopes against the literature before a real campaign.

## 3. Workflow architecture

```mermaid
flowchart LR
    T["Target RBD + hotspots
(30_SC2RBD)"] --> G["Proteina-Complexa
co-design seq+structure
BROAD pool, fast (no AF2)"]
    G --> H["Boltz-2 holo co-fold
each binder + target"]
    H --> V["Rank by co-folding score
ipTM (+ complex/binder pLDDT)"]
    G -. optional .-> A["Boltz-2 apo refold
binder alone -> stability RMSD"]
    A -. optional .-> V
    V --> R["Top-K binders
ranked_binders.json + plots + Mol* view"]
```

**Strategy:** generate a **broad pool cheaply** (no AF2 reward), **co-fold every candidate with
Boltz-2**, and **rank by co-folding confidence** (ipTM, with complex/binder pLDDT as support). The
strict gate (ipSAE ≥ 0.45, ipTM ≥ 0.65, pLDDTs ≥ 0.70, apo↔holo RMSD ≤ 2.5 Å, ≥ 20% hotspot contact)
is reported as context; the **deliverable is the ranked top-K**. More samples → a better top of the
ranking. Reward-guided best-of-n + AF2 is available (`PBD_ALGORITHM=best-of-n PBD_AF2_REWARD=1`) but
is slower and, at demo scale, rarely beats simply co-folding a larger pool.

## 4. Before you run

This notebook expects the launchable environment (set up by `setup.sh`): Proteina-Complexa built at
`$COMPLEXA_REPO`, the agent-toolkit skill at `$COMPLEXA_SKILL`, AF2 params at `$AF2_DIR`, and a
**local Boltz-2 NIM** on `$BOLTZ2_URL` (`http://localhost:8000/...`).

> **Select the `Python (Complexa)` kernel** (top-right kernel picker, or *Kernel → Change Kernel*).
> `setup.sh` registers it, pointing at the Proteina-Complexa `.venv` with all deps + env baked in.
> The default `Python 3` kernel is a different environment and will fail with `No module named numpy`.

**Budget & quality.** `DEMO=1` (default) generates `PBD_NUM_SAMPLES=48` (no AF2) and co-folds
`PBD_N_ASSESS=24` with Boltz-2 at `PBD_DIFFUSION_SAMPLES=5` samples/design (best kept) and
`PBD_SAMPLING_STEPS=75` refinement steps — a quality-leaning config (~20–40 min; longer on smaller
GPUs like L40S). The co-fold cost scales with `N_ASSESS × DIFFUSION_SAMPLES × SAMPLING_STEPS`, so
**dial those down for speed** or up for quality. `OPENHACKATHON_DEMO_MODE=0` uses the max
(96 / 64 / 8 / 100). Enable the apo stability check with `PBD_APO=1`.

**GPU.** Runs on **A100 / H100 / L40S**. On the 48 GB **L40S** (vs 80 GB A100/H100), very large
co-folding jobs — big complexes, high `PBD_DIFFUSION_SAMPLES`, or the AF2-reward path — can hit
**CUDA out-of-memory (OOM)**. If that happens, reduce `PBD_N_ASSESS` / `PBD_DIFFUSION_SAMPLES` (or
use a smaller binder/target).

**Responsible use:** de novo binder design is dual-use — legitimate research/therapeutic intent only.

In [ ]:
import os, sys, json, subprocess, pathlib, datetime, glob, textwrap

# --- environment (from setup.sh / ~/.complexa_env; fall back to known paths) ---
HOME = pathlib.Path.home()
COMPLEXA_REPO = os.environ.get("COMPLEXA_REPO", str(HOME / "Proteina-Complexa"))
COMPLEXA_SKILL = os.environ.get(
    "COMPLEXA_SKILL",
    str(HOME / "bionemo-agent-toolkit/workflows/generative_protein_binder_design/complexa-binder-design"),
)
AF2_DIR = os.environ.get("AF2_DIR", str(pathlib.Path(COMPLEXA_SKILL) / "community_models/ckpts/AF2"))
BOLTZ2_URL = os.environ.get("BOLTZ2_URL", "http://localhost:8000/biology/mit/boltz2/predict")

# --- campaign config ---
DEMO = os.environ.get("OPENHACKATHON_DEMO_MODE", "1") != "0"
TARGET       = os.environ.get("PBD_TARGET", "30_SC2RBD")
# Strategy: generate a BROAD pool fast (no AF2), then let Boltz-2 co-folding score + rank them.
ALGORITHM    = os.environ.get("PBD_ALGORITHM", "single-pass")    # broad sampling (no reward search)
AF2_REWARD   = os.environ.get("PBD_AF2_REWARD", "0") != "0"      # default OFF: no AF2 (Boltz-2 is the judge)
NUM_SAMPLES  = int(os.environ.get("PBD_NUM_SAMPLES", "48" if DEMO else "96"))    # generated pool
N_ASSESS     = int(os.environ.get("PBD_N_ASSESS", "24" if DEMO else "64"))       # how many to co-fold with Boltz-2
DIFFUSION_SAMPLES = int(os.environ.get("PBD_DIFFUSION_SAMPLES", "5" if DEMO else "8"))   # Boltz-2 samples/design; keep best
SAMPLING_STEPS    = int(os.environ.get("PBD_SAMPLING_STEPS", "75" if DEMO else "100"))   # Boltz-2 refinement steps
WITH_APO     = os.environ.get("PBD_APO", "0" if DEMO else "1") != "0"        # apo adds stability RMSD (2x calls)
TOP_K        = int(os.environ.get("PBD_TOP_K", "5"))
SEED         = int(os.environ.get("PBD_SEED", "0"))
RUN = f"nb_{datetime.datetime.now():%Y%m%d_%H%M%S}"
RUN_DIR = pathlib.Path(COMPLEXA_SKILL) / "outputs" / RUN
RUN_DIR.mkdir(parents=True, exist_ok=True)

def sh(cmd, timeout=5400, stream=True):
    """Run a bash command with the Complexa venv + Hydra env sourced."""
    full = (
        f'cd "{COMPLEXA_REPO}" && source .venv/bin/activate && source env.sh 2>/dev/null; '
        f'export COMPLEXA_REPO="{COMPLEXA_REPO}" AF2_DIR="{AF2_DIR}" BOLTZ2_URL="{BOLTZ2_URL}" '
        # keep JAX/AF2 from preallocating the whole GPU so it coexists with the resident Boltz-2 NIM
        f'XLA_PYTHON_CLIENT_PREALLOCATE=false XLA_PYTHON_CLIENT_MEM_FRACTION=0.7; '
        f'cd "{COMPLEXA_SKILL}"; {cmd}'
    )
    p = subprocess.run(["bash", "-lc", full], capture_output=True, text=True, timeout=timeout)
    out = (p.stdout or "") + (p.stderr or "")
    if stream:
        print(out[-4000:])
    return p.returncode, out

for pkg in ("ipymolstar", "py3Dmol"):
    try: __import__(pkg)
    except Exception: subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
import numpy as np, pandas as pd, requests
import matplotlib.pyplot as plt

print(f"DEMO={DEMO} TARGET={TARGET} ALGORITHM={ALGORITHM} AF2_REWARD={AF2_REWARD} "
      f"NUM_SAMPLES={NUM_SAMPLES} N_ASSESS={N_ASSESS} DIFFUSION_SAMPLES={DIFFUSION_SAMPLES} "
      f"WITH_APO={WITH_APO} TOP_K={TOP_K}")
print(f"COMPLEXA_REPO={COMPLEXA_REPO}\nCOMPLEXA_SKILL={COMPLEXA_SKILL}\nBOLTZ2_URL={BOLTZ2_URL}")
print(f"RUN_DIR={RUN_DIR}")

In [ ]:
# Health checks: Complexa CLI + target, local Boltz-2 NIM, AF2 params
rc, out = sh('cd "$COMPLEXA_REPO" && complexa target show %s 2>&1 | sed -n "1,20p"' % TARGET, timeout=120)
b2_ready = False
try:
    r = requests.get(BOLTZ2_URL.replace('/biology/mit/boltz2/predict', '/v1/health/ready'), timeout=8)
    b2_ready = r.status_code == 200
except Exception as e:
    print("boltz2 health error:", e)
print(f"Boltz-2 ready: {b2_ready}")
print(f"AF2 params present: {pathlib.Path(AF2_DIR, 'params').exists()}")

## 5. Stage 1 — target and hotspots

Read the registered target (span, hotspots, binder length), fetch the RBD structure, and show the
conditioned hotspot patch with **Mol\***.

In [ ]:
# Parse target metadata from `complexa target show`
rc, out = sh(f'cd "$COMPLEXA_REPO" && complexa target show {TARGET}', timeout=120, stream=False)
def _field(name):
    for line in out.splitlines():
        if name in line and ":" in line:
            return line.split(":", 1)[1].strip()
    return ""
pdb_id = (_field("pdb_id") or "6m0j").lower()
target_span = _field("target_input")           # e.g. E333-526
import ast
try:
    hotspots = ast.literal_eval(_field("hotspot_residues") or "[]")
except Exception:
    hotspots = []
hotspot_resnums = [int(''.join(ch for ch in h if ch.isdigit())) for h in hotspots]
target_chain_hint = (hotspots[0][0] if hotspots else "E")
print(f"target={TARGET} pdb={pdb_id} span={target_span} hotspots={hotspots} binder chain from Complexa is typically 'B'")

# Fetch the full experimental structure for a target view (RCSB)
pdb_text = requests.get(f"https://files.rcsb.org/download/{pdb_id.upper()}.pdb", timeout=60).text
open(RUN_DIR / f"{pdb_id}.pdb", "w").write(pdb_text)

In [ ]:
# Mol* view of the target with hotspots highlighted (red)
from ipymolstar import PDBeMolstar
from ipywidgets import Layout
color = [{"struct_asym_id": target_chain_hint, "color": "#9ecae1"}]
for r in hotspot_resnums:
    color.append({"struct_asym_id": target_chain_hint, "residue_number": int(r), "color": "#e53935"})
tv = PDBeMolstar(custom_data={"data": pdb_text, "format": "pdb", "binary": False},
                 hide_water=True, hide_controls_icon=True, hide_expand_icon=True,
                 hide_settings_icon=True, hide_animation_icon=True,
                 layout=Layout(width="600px", height="600px"))  # square viewer
tv.color_data = {"data": color, "nonSelectedColor": None}
tv

## 6. Stage 2 — generate a broad pool of binders (Proteina-Complexa)

Co-design binder sequence + structure. We default to fast **`single-pass` sampling with no AF2**
(`AF2_REWARD=0`) to produce a **large pool** quickly — Boltz-2 does the scoring next. For
reward-guided generation instead, set `PBD_ALGORITHM=best-of-n PBD_AF2_REWARD=1` (slower, uses AF2).

In [ ]:
override = "" if AF2_REWARD else "--override '~generation.reward_model.reward_models.af2folding'"
gen_cmd = (
    f"python scripts/complexa_design.py run --mode generate "
    f"--task-name {TARGET} --run-name {RUN} --algorithm {ALGORITHM} "
    f"--num-samples {NUM_SAMPLES} --seed {SEED} {override} --out '{RUN_DIR}/inference'"
)
print(gen_cmd, "\n")
rc, out = sh(gen_cmd, timeout=5400)
# discover generated complex PDBs
pdbs = sorted(glob.glob(f"{COMPLEXA_REPO}/inference/*{TARGET}*{RUN}*/**/*.pdb", recursive=True))
if not pdbs:
    pdbs = sorted(glob.glob(f"{COMPLEXA_REPO}/inference/**/*.pdb", recursive=True),
                  key=os.path.getmtime)[-NUM_SAMPLES:]
print(f"generated {len(pdbs)} complex PDBs (rc={rc})")
for p in pdbs[:6]:
    print("  ", p)

## 7. Stage 3 — co-fold assessment with Boltz-2

Co-fold each candidate complex with the **local Boltz-2 NIM** (holo binder+target) to get an
**independent** interface confidence (ipTM) plus complex/binder pLDDT, ipSAE, and hotspot contact.
Boltz-2's diffusion is stochastic, so we draw **`DIFFUSION_SAMPLES` structures per design and keep the
best-ipTM one** — this cuts the run-to-run noise. The apo (binder-alone) refold that adds an apo↔holo
stability RMSD is **optional** (`PBD_APO=1`); the demo skips it to co-fold a larger pool in the same
time budget.

In [ ]:
# hotspots.json in the validator schema: {"hotspot_residues":[{"position": <refold index>, ...}]}.
# Boltz-2 renumbers the refolded target chain 1..N, so map author residue -> index via span start.
import re as _re
_m = _re.search(r"(\d+)", target_span or "")
span_start = int(_m.group(1)) if _m else 1
hs_entries = [{"position": rn - span_start + 1, "chain": "A", "author": h}
              for rn, h in zip(hotspot_resnums, hotspots)]
hotspots_json = RUN_DIR / "hotspots.json"
json.dump({"hotspot_residues": hs_entries}, open(hotspots_json, "w"))
print("hotspot refold positions:", [e["position"] for e in hs_entries])
shortlist = pdbs[:N_ASSESS]
# Stage 3a: independent Boltz-2 holo co-fold in the notebook. For each design we request
# DIFFUSION_SAMPLES structures and keep the BEST-ipTM one (reduces Boltz-2's sampling noise).
# Reuse the skill's chain parser + retrying POST so the payload matches the validated path.
import sys as _sys
_sys.path.insert(0, str(pathlib.Path(COMPLEXA_SKILL) / "scripts"))
import boltz2_refold as _b2r
raw_dir = RUN_DIR / "validation" / "raw"; raw_dir.mkdir(parents=True, exist_ok=True)
_PERSAMPLE = ("structures","iptm_scores","ptm_scores","complex_plddt_scores","complex_iplddt_scores",
              "protein_iptm_scores","ligand_iptm_scores","complex_pde_scores","complex_ipde_scores",
              "chains_ptm_scores","pair_chains_iptm_scores","pae","pde","confidence_scores","affinities")
n_ok = 0
for i, pdb in enumerate(shortlist):
    seqs = _b2r.chain_seqs(pdb); tgt, bnd = seqs.get("A"), seqs.get("B")
    if not tgt or not bnd:
        print(f"[skip] cand{i:02d}: chains {list(seqs)}"); continue
    body = {"polymers": [{"id": "A", "molecule_type": "protein", "sequence": tgt},
                         {"id": "B", "molecule_type": "protein", "sequence": bnd}],
            "recycling_steps": 3, "sampling_steps": SAMPLING_STEPS,
            "diffusion_samples": DIFFUSION_SAMPLES, "step_scale": 1.638,
            "output_format": "mmcif", "write_full_pae": True}
    try:
        resp = _b2r.post_with_retry(BOLTZ2_URL, body, {"Content-Type": "application/json"}, max_retries=5)
    except Exception as e:
        print(f"[holo] cand{i:02d} FAILED: {e}"); continue
    ipts = resp.get("iptm_scores") or []
    if len(ipts) > 1:                       # move the best-ipTM sample to index 0 (downstream reads [0])
        bi = max(range(len(ipts)), key=lambda k: ipts[k])
        for key in _PERSAMPLE:
            v = resp.get(key)
            if isinstance(v, list) and len(v) == len(ipts):
                resp[key] = [v[bi]] + [v[k] for k in range(len(v)) if k != bi]
    (raw_dir / f"cand{i:02d}.json").write_text(json.dumps(resp))
    best = max(ipts) if ipts else None
    print(f"[holo] cand{i:02d}: best ipTM={round(best,3) if isinstance(best,(int,float)) else '?'} "
          f"over {len(ipts) or 1} sample(s)")
    n_ok += 1
print(f"=== {n_ok} complexes co-folded (diffusion_samples={DIFFUSION_SAMPLES}) ===")

# Stage 3b: score + rank from the co-folds; apo (binder-alone) stability check is optional
apo_flag = "" if WITH_APO else "--no-apo"
score_cmd = (
    f"python scripts/validate_binders.py --run-dir '{RUN_DIR}' --endpoint local "
    f"--target-chain A --binder-chain B --hotspots '{hotspots_json}' {apo_flag}"
)
print(score_cmd, "\n")
rc, out = sh(score_cmd, timeout=10800)
print("assessment rc=", rc)

## 8. Stage 4 — rank by Boltz-2 co-folding score, report the top-K

This is where the broad pool pays off: we **rank all co-folded designs by Boltz-2 interface
confidence** (`ipTM`, with `complex_plddt` / `binder_plddt` as supporting evidence) and surface the
**top-K binders**. The strict pass/fail gate (ipTM ≥ 0.65, etc.) is reported as *context*, but the
deliverable is the **ranked shortlist** — the highest-confidence complexes to carry forward. Against
a hard target like the RBD you often won't clear the absolute bar at demo scale; what matters is that
Boltz-2, an **independent** model, consistently separates the better designs, and that scaling
`PBD_NUM_SAMPLES` / `PBD_N_ASSESS` (or `OPENHACKATHON_DEMO_MODE=0`) pushes the top of the ranking up.

In [ ]:
ranked_path = None
for cand in [RUN_DIR / "ranked_binders.json", RUN_DIR / "validation_scores.json"]:
    if cand.exists(): ranked_path = cand; break
if ranked_path is None:
    hits = glob.glob(f"{RUN_DIR}/**/ranked_binders.json", recursive=True)
    ranked_path = pathlib.Path(hits[0]) if hits else None
print("ranked file:", ranked_path)

rows = json.load(open(ranked_path)) if ranked_path else []
if isinstance(rows, dict):
    rows = rows.get("binders") or rows.get("designs") or list(rows.values())

def g(d, *ks):
    for k in ks:
        if isinstance(d, dict) and d.get(k) is not None: return d[k]
    return None
recs = []
for d in rows:
    recs.append({
        "id": g(d, "id", "design_id", "name"),
        "iptm": g(d, "iptm", "ipTM"),
        "ipsae_min": g(d, "ipsae_min", "ipsae"),
        "complex_plddt": g(d, "complex_plddt"),
        "binder_plddt": g(d, "binder_plddt"),
        "apo_binder_plddt": g(d, "apo_binder_plddt"),
        "binder_rmsd": g(d, "binder_rmsd", "apo_holo_rmsd"),
        "hotspot_contact": g(d, "hotspot_contact_frac", "hotspot_contact"),
        "pass": g(d, "pass", "passed"),
        "failure_reason": g(d, "failure_reason"),
        "holo_cif": g(d, "holo_cif", "complex_cif", "holo"),
    })
df = pd.DataFrame(recs)
_r = lambda v: round(v, 3) if isinstance(v, (int, float)) else v
# rank by Boltz-2 co-folding confidence: ipTM primary, complex/binder pLDDT as tiebreakers
if not df.empty:
    df["cofold_score"] = df["iptm"].fillna(0)
    df = df.sort_values(["iptm", "complex_plddt", "binder_plddt"],
                        ascending=False, na_position="last").reset_index(drop=True)
    df.insert(0, "rank", range(1, len(df) + 1))
    # co-folding gate — evaluable from the holo co-fold alone (no apo needed):
    # confident interface + confident fold + hotspot engagement
    ge = lambda s, t: df[s].fillna(0) >= t
    df["cofold_pass"] = (ge("iptm", 0.65) & ge("complex_plddt", 0.70)
                         & ge("binder_plddt", 0.70) & ge("hotspot_contact", 0.20))
n_cofold = int(df["cofold_pass"].sum()) if "cofold_pass" in df and not df.empty else 0
n_pass = int(df["pass"].fillna(False).astype(bool).sum()) if "pass" in df and not df.empty else 0
best = df.iloc[0].to_dict() if not df.empty else {}
msg = (f"designs assessed: {len(df)}   |   pass co-folding gate "
       f"(ipTM>=0.65, complex&binder pLDDT>=0.70, hotspot>=0.20): {n_cofold}")
if WITH_APO: msg += f"   |   pass full strict gate (incl. apo): {n_pass}"
print(msg)
if best:
    print(f"top binder: {best.get('id')}  ipTM={_r(best.get('iptm'))}  "
          f"complex_plddt={_r(best.get('complex_plddt'))}  binder_plddt={_r(best.get('binder_plddt'))}  "
          f"hotspot={best.get('hotspot_contact')}  cofold_pass={best.get('cofold_pass')}")
show_cols = [c for c in ["rank","id","iptm","complex_plddt","binder_plddt","hotspot_contact","cofold_pass"]
             + (["apo_binder_plddt","binder_rmsd","pass"] if WITH_APO else []) if c in df.columns]
print(f"\nTop {TOP_K} binders by Boltz-2 co-folding confidence:")
df.head(TOP_K)[show_cols] if not df.empty else print("no assessed designs")

In [ ]:
# Assessment plots (Boltz-2 co-folding based)
if not df.empty:
    fig, ax = plt.subplots(2, 2, figsize=(14, 10))
    p = df["cofold_pass"] if "cofold_pass" in df else pd.Series([False]*len(df))
    # 1) interface vs binder confidence
    ax[0,0].scatter(df.loc[~p,"iptm"], df.loc[~p,"binder_plddt"], c="#bbbbbb", edgecolor="k", label="below co-fold gate", s=70)
    ax[0,0].scatter(df.loc[p,"iptm"],  df.loc[p,"binder_plddt"],  c="#76b900", edgecolor="k", label="passes co-fold gate", s=90)
    ax[0,0].axvline(0.65, ls="--", c="r", alpha=.5); ax[0,0].axhline(0.70, ls="--", c="r", alpha=.5)
    ax[0,0].set_xlabel("Boltz-2 ipTM"); ax[0,0].set_ylabel("binder pLDDT")
    ax[0,0].set_title("Interface vs binder confidence"); ax[0,0].legend()
    # 2) co-folding ipTM distribution across the pool
    ax[0,1].hist(df["iptm"].dropna(), bins=10, color="#76b900", edgecolor="k")
    ax[0,1].axvline(0.65, ls="--", c="r"); ax[0,1].set_title("Co-folding ipTM distribution"); ax[0,1].set_xlabel("ipTM")
    # 3) top-K ranked by ipTM
    topk = df.head(TOP_K)
    ax[1,0].barh(topk["id"].astype(str)[::-1], topk["iptm"][::-1], color="#76b900", edgecolor="k")
    ax[1,0].set_xlabel("Boltz-2 ipTM"); ax[1,0].set_title(f"Top {TOP_K} binders (ranked)")
    # 4) funnel
    funnel = {"generated": len(pdbs), "co-folded": len(df), "pass co-fold gate": n_cofold}
    b = ax[1,1].bar(list(funnel), list(funnel.values()), color="#76b900"); ax[1,1].bar_label(b)
    ax[1,1].set_title("Design funnel"); ax[1,1].set_ylabel("count")
    plt.tight_layout(); plt.show()

## 9. Stage 5 — Mol\* view of the #1 ranked binder–target complex

The top binder by Boltz-2 co-folding ipTM: binder (orange) docked onto the RBD (gray), with
conditioned hotspots (red). A good design wraps the binder over the hotspot patch with high
interface confidence.

In [ ]:
# Boltz-2 embeds the holo complex CIF inside validation/raw/<id>.json (chain A = target/RBD,
# chain B = designed binder). The apo CIFs are the binder alone, so pull the holo complex here.
mol = None
top = df.iloc[0].to_dict() if not df.empty else {}
tid = top.get("id")
cif_text = None
raw_json = RUN_DIR / "validation" / "raw" / f"{tid}.json"
if raw_json.exists():
    structs = json.load(open(raw_json)).get("structures") or []
    if structs:
        s0 = structs[0]
        cif_text = s0.get("structure") if isinstance(s0, dict) else s0
print("top design:", tid, "ipTM=", top.get("iptm"),
      "holo complex:", str(raw_json) if cif_text else "not found")
if cif_text:
    from ipymolstar import PDBeMolstar
    from ipywidgets import Layout
    mol = PDBeMolstar(custom_data={"data": cif_text, "format": "cif", "binary": False},
                      hide_water=True, hide_controls_icon=True, hide_expand_icon=True,
                      hide_settings_icon=True, hide_animation_icon=True,
                      layout=Layout(width="600px", height="600px"))  # square viewer
    # target chain A gray, designed binder chain B orange, conditioned hotspots red
    cdata = [{"struct_asym_id": "A", "color": "#cfd8dc"},
             {"struct_asym_id": "B", "color": "#ff8c00"}]
    for e in hs_entries:
        cdata.append({"struct_asym_id": "A", "residue_number": int(e["position"]), "color": "#e53935"})
    mol.color_data = {"data": cdata, "nonSelectedColor": None}
mol

## 10. Report and next steps

- **Scale up for more reliable hits:** raise `PBD_NUM_SAMPLES` / `PBD_N_ASSESS` (bigger pool) and
  `PBD_DIFFUSION_SAMPLES` (e.g. 5 — averages out Boltz-2's co-folding noise), or set
  `OPENHACKATHON_DEMO_MODE=0` (64 / 48 / 5). Add `PBD_APO=1` for the apo↔holo stability check, or try
  reward-guided generation with `PBD_ALGORITHM=best-of-n PBD_AF2_REWARD=1`.
- **Other targets:** set `PBD_TARGET` to any registered target (`complexa target list`) — PD-L1,
  IL-17A, TNFα, VEGFA, HER2, etc. — or register your own with `complexa target add`.
- **Outputs:** `RUN_DIR/` holds `ranked_binders.json/.csv`, the Boltz-2 `validation/`, and the
  generated complexes.
- **Independent validation** (Boltz-2 ≠ Proteina-Complexa) is the point: rank by interface
  confidence + stability and report a success rate vs the gate.

**Responsible use:** keep de novo binder design to legitimate research and therapeutic intent.